## Perch v2 TFLite Conversion Test (v2 — with Optimize.DEFAULT)

Convert `perch_v2_cpu` SavedModel → TFLite with `Optimize.DEFAULT` (float16 weight quantization).

**Key finding from BirdCLEF 2025 paper (DS@GT)**: their 10× speedup (17s → 1.4s/file) used
`converter.optimizations = [tf.lite.Optimize.DEFAULT]` — weight quantization shrinks the model
and enables faster CPU kernels. Our v1 test omitted this flag and got 0.9× (actually slower).
v2 adds it to test whether quantization gives speedup despite `SELECT_TF_OPS` fallback.

If model size drops from ~407 MB → ~100-200 MB, quantization applied successfully.
If speedup ≥ 3×: Perch TFLite is viable for blend kernel.

In [ ]:
# Cell 1 — Install TF (offline)
import os
import subprocess
import sys
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = ""

_WHL = Path("/kaggle/input/notebooks/kdmitrie/bc26-tensorflow-2-20-0/wheel")
if _WHL.exists():
    whl = sorted(_WHL.glob("tensorflow*.whl"))
    if whl:
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                str(whl[0]),
                "-q",
                "--no-deps",
                "--force-reinstall",
            ],
            check=True,
        )
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "keras",
                "tf_keras",
                "-q",
                "--no-deps",
            ],
            check=False,
        )
        print(f"Installed TF from {whl[0].name}", flush=True)

import tensorflow as tf

print(f"TensorFlow: {tf.__version__}", flush=True)

In [ ]:
# Cell 2 — Load SavedModel + inspect
import time

import numpy as np

MODEL_DIR = Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1")
print(f"Model dir exists: {MODEL_DIR.exists()}", flush=True)

t0 = time.time()
birdclassifier = tf.saved_model.load(str(MODEL_DIR))
infer_fn = birdclassifier.signatures["serving_default"]
print(f"Loaded in {time.time() - t0:.1f}s", flush=True)
print(f"Input keys:  {list(infer_fn.structured_input_signature[1].keys())}", flush=True)
print(f"Output keys: {list(infer_fn.structured_outputs.keys())}", flush=True)

dummy = tf.zeros([1, 160000], dtype=tf.float32)
out = infer_fn(inputs=dummy)
print(
    f"logits shape: {out['label'].shape}  emb shape: {out['embedding'].shape}",
    flush=True,
)

In [ ]:
# Cell 3 — TFLite conversion with Optimize.DEFAULT (v2)
#
# DS@GT BirdCLEF 2025 used:
#   converter.optimizations = [tf.lite.Optimize.DEFAULT]  ← key ingredient we missed
#   converter.target_spec.supported_ops = [TFLITE_BUILTINS, SELECT_TF_OPS]
# This applies float16 weight quantization even when SELECT_TF_OPS is needed.
# Result indicator: if model size drops from 407 MB → <200 MB, quantization worked.
import time

print("Attempting TFLite conversion with Optimize.DEFAULT...", flush=True)
t0 = time.time()
tflite_model = None
try:
    converter = tf.lite.TFLiteConverter.from_saved_model(str(MODEL_DIR))
    # KEY: weight quantization — this was missing from v1 test
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS,
        tf.lite.OpsSet.SELECT_TF_OPS,
    ]
    converter.allow_custom_ops = True
    tflite_model = converter.convert()
    t_conv = time.time() - t0
    size_mb = len(tflite_model) / 1e6
    print(f"Conversion SUCCESS in {t_conv:.1f}s", flush=True)
    print(
        f"TFLite model size: {size_mb:.1f} MB  (v1 was 407 MB — smaller = quantization worked)",
        flush=True,
    )
    out_path = Path("/kaggle/working/perch_v2_cpu_quantized.tflite")
    out_path.write_bytes(tflite_model)
    print(f"Saved: {out_path}", flush=True)
    if size_mb < 300:
        print(">>> Model shrank — float16 quantization applied to weights!", flush=True)
    else:
        print(
            ">>> Model size similar to v1 — quantization may not have applied to SELECT_TF_OPS kernels",
            flush=True,
        )
except Exception as e:
    print(f"Conversion FAILED: {type(e).__name__}: {e}", flush=True)

In [ ]:
# Cell 4 — Load 20 training soundscapes for benchmarking
import librosa

DATA = Path("/kaggle/input/competitions/birdclef-2026")
test_files = sorted((DATA / "train_soundscapes").glob("*.ogg"))[:20]
print(f"Loading {len(test_files)} soundscapes...", flush=True)

SR = 32000
CLIP = SR * 5  # 5s window
N_WINDOWS = 12  # 60s / 5s

audios = []
for p in test_files:
    y, _ = librosa.load(p, sr=SR, mono=True, duration=60.0)
    if len(y) < SR * 60:
        y = np.pad(y, (0, SR * 60 - len(y)))
    for i in range(N_WINDOWS):
        audios.append(y[i * CLIP : (i + 1) * CLIP].astype(np.float32))

print(f"Total windows: {len(audios)} ({len(test_files)} files × {N_WINDOWS})", flush=True)

In [ ]:
# Cell 5 — Benchmark SavedModel
BATCH = 16


def infer_savedmodel(windows):
    results = []
    for i in range(0, len(windows), BATCH):
        batch = tf.constant(np.stack(windows[i : i + BATCH]), dtype=tf.float32)
        out = infer_fn(inputs=batch)
        results.append(out["label"].numpy())
    return np.concatenate(results)


# Warm up XLA
print("Warming up SavedModel (XLA compile)...", flush=True)
_ = infer_savedmodel(audios[:BATCH])

print("Benchmarking SavedModel...", flush=True)
t0 = time.time()
sm_preds = infer_savedmodel(audios)
t_sm = time.time() - t0
per_file_sm = t_sm / len(test_files)
print(f"SavedModel: {t_sm:.1f}s total | {per_file_sm:.2f}s/file", flush=True)
print(f"Extrapolated 739 files: {per_file_sm * 739 / 60:.1f} min", flush=True)

In [ ]:
# Cell 6 — Benchmark TFLite (if conversion succeeded)
# Notes from v2:
# - ai_edge_litert not installable without internet on Kaggle
# - tf.lite.Interpreter fallback works (allocate_tensors OK, no resize needed)
# - TFLite output names are 'StatefulPartitionedCall:N', NOT 'label'/'embedding'
#   → find label output by shape: largest last dim = 14795 logits
if tflite_model is None:
    print("Skipping — TFLite conversion failed")
else:
    interp = tf.lite.Interpreter(model_content=tflite_model, num_threads=2)
    inp_det = interp.get_input_details()
    out_det = interp.get_output_details()
    print(f"TFLite input:  {inp_det[0]['shape']} dtype={inp_det[0]['dtype']}", flush=True)
    for i, d in enumerate(out_det):
        print(f"  output[{i}]: name={d['name']}  shape={d['shape']}", flush=True)

    # Model has fixed input shape [1, 160000] — no resize needed
    interp.allocate_tensors()

    # Identify label output by shape: largest last dimension = 14795 logits
    label_idx = max(range(len(out_det)), key=lambda i: out_det[i]["shape"][-1])
    print(
        f"Label output idx={label_idx}  name={out_det[label_idx]['name']}  shape={out_det[label_idx]['shape']}",
        flush=True,
    )

    try:

        def infer_tflite_single(windows):
            results = []
            for w in windows:
                interp.set_tensor(inp_det[0]["index"], w.reshape(1, -1))
                interp.invoke()
                results.append(interp.get_tensor(out_det[label_idx]["index"]).copy())
            return np.concatenate(results)

        # Warm up
        print("Warming up TFLite...", flush=True)
        _ = infer_tflite_single(audios[:2])

        # 1-file test first
        print("Benchmarking TFLite (1 file = 12 windows)...", flush=True)
        t0 = time.time()
        _ = infer_tflite_single(audios[:N_WINDOWS])
        t_1file = time.time() - t0
        print(
            f"1 file: {t_1file:.2f}s  (extrapolated 739 files: {t_1file * 739 / 60:.1f} min)",
            flush=True,
        )

        # Full benchmark
        print("Benchmarking TFLite (all 20 files)...", flush=True)
        t0 = time.time()
        tfl_preds = infer_tflite_single(audios)
        t_tfl = time.time() - t0
        per_file_tfl = t_tfl / len(test_files)
        print(f"TFLite: {t_tfl:.1f}s total | {per_file_tfl:.2f}s/file", flush=True)
        print(f"Extrapolated 739 files: {per_file_tfl * 739 / 60:.1f} min", flush=True)

        # Correctness check
        max_diff = np.abs(sm_preds - tfl_preds).max()
        mean_diff = np.abs(sm_preds - tfl_preds).mean()
        print(
            f"Max diff vs SavedModel: {max_diff:.6f}  mean diff: {mean_diff:.6f}",
            flush=True,
        )
        print(f"SPEEDUP: {t_sm / t_tfl:.1f}x", flush=True)

    except Exception as e:
        import traceback

        print(f"TFLite inference failed: {type(e).__name__}: {e}", flush=True)
        traceback.print_exc()

In [ ]:
# Cell 7 — Summary
print("=" * 50)
print("SUMMARY")
print("=" * 50)
print(f"SavedModel: {per_file_sm:.2f}s/file → {per_file_sm * 739 / 60:.1f} min for 739 files")
if tflite_model is not None:
    try:
        print(f"TFLite:     {per_file_tfl:.2f}s/file → {per_file_tfl * 739 / 60:.1f} min for 739 files")
        print(f"Speedup:    {t_sm / t_tfl:.1f}x")
        print(f"Max logit diff: {max_diff:.6f}")
        print(f"TFLite model: {len(tflite_model) / 1e6:.1f} MB")
    except NameError:
        print("TFLite inference failed — see Cell 6")
else:
    print("TFLite: CONVERSION FAILED — SavedModel uses unsupported ops")
    print("Alternative: try tf2onnx → ONNX Runtime")